# Aula 15 — Laboratório: testes de hipótese, p-value e poder

Este laboratório reproduz as ideias centrais da aula com simulações e testes verificáveis.

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/02-statistics/notebooks/15-testes-pvalue-poder-laboratorio.ipynb)

**Dependências:** Python 3.10+, NumPy, pandas, SciPy e Matplotlib.  
**Reprodutibilidade:** semente fixa \`20260907\`.  
**Método:** hipóteses, direção, nível de significância e regra de parada são definidos antes de observar os resultados.

## Objetivos

Ao final, você terá:

1. calculado e conferido um teste t de uma amostra;
2. visualizado o p-value como área de cauda sob \(H_0\);
3. verificado por simulação que \(\alpha\) calibra o erro tipo I;
4. relacionado tamanho amostral, efeito e poder;
5. preservado o pareamento ao comparar modelos;
6. medido como a parada opcional infla falsos positivos.

> Um p-value não é \(P(H_0\mid dados)\). Ele é calculado supondo \(H_0\) e mede a extremidade dos dados segundo o procedimento especificado.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from IPython.display import display
from scipy.stats import nct, t, ttest_1samp, ttest_ind, ttest_rel

SEED = 20260907
ALPHA = 0.05

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seed:", SEED)

## 1. Teste t unilateral para a latência média

Pergunta definida antes dos dados:

\[
H_0:\mu=100\text{ ms}
\qquad\text{versus}\qquad
H_1:\mu<100\text{ ms}.
\]

Como o desvio-padrão populacional é desconhecido, usamos

\[
t=\frac{\bar x-\mu_0}{s/\sqrt n},
\]

com \(n-1\) graus de liberdade. A alternativa é unilateral à esquerda porque o critério operacional era “abaixo de 100 ms”.

In [ ]:
latencias = np.array([92, 88, 95, 101, 97, 90, 93, 105, 99, 94, 96, 91], dtype=float)
mu0 = 100.0
n = latencias.size

media = latencias.mean()
desvio = latencias.std(ddof=1)
erro_padrao = desvio / np.sqrt(n)
t_manual = (media - mu0) / erro_padrao
p_manual = t.cdf(t_manual, df=n - 1)

resultado = ttest_1samp(latencias, popmean=mu0, alternative="less")

resumo = pd.DataFrame({
    "quantidade": ["n", "média", "desvio-padrão", "erro-padrão", "t observado", "p-value unilateral"],
    "valor": [n, media, desvio, erro_padrao, t_manual, p_manual],
})
display(resumo.round({"valor": 9}))

assert np.isclose(t_manual, resultado.statistic, rtol=0, atol=1e-12)
assert np.isclose(p_manual, resultado.pvalue, rtol=0, atol=1e-12)
assert p_manual < ALPHA

print(f"Decisão a α={ALPHA:.2f}: rejeitar H0.")
print("Interpretação: os dados são incompatíveis com μ=100 ms na direção pré-especificada.")

In [ ]:
gl = n - 1
x = np.linspace(-5.5, 5.5, 1_500)
densidade = t.pdf(x, df=gl)
mascara = x <= t_manual

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, densidade, color="#1f4e79", linewidth=2, label=f"t({gl}) sob H₀")
ax.fill_between(x[mascara], densidade[mascara], color="#e45756", alpha=0.65,
                label=f"cauda observada: p = {p_manual:.4f}")
ax.axvline(t_manual, color="#8b1e3f", linestyle="--", label=f"t = {t_manual:.3f}")
ax.set(xlabel="Estatística t", ylabel="Densidade",
       title="p-value unilateral: área tão ou mais extrema sob H₀")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 2. \(\alpha\) como calibração de longo prazo

Sob \(H_0\), um procedimento de nível 5% deve rejeitar aproximadamente 5% das vezes **se suas hipóteses e sua regra de análise forem respeitadas**. Vamos repetir 50 mil experimentos independentes com \(n=12\), média 100 e desvio-padrão 5.

In [ ]:
rng = np.random.default_rng(SEED)
repeticoes = 50_000
n_h0 = 12

amostras_h0 = rng.normal(loc=100.0, scale=5.0, size=(repeticoes, n_h0))
testes_h0 = ttest_1samp(amostras_h0, popmean=100.0, axis=1, alternative="less")
taxa_erro_tipo_i = np.mean(testes_h0.pvalue < ALPHA)

print(f"Repetições: {repeticoes:,}".replace(",", "."))
print(f"α planejado: {ALPHA:.4f}")
print(f"Taxa simulada de erro tipo I: {taxa_erro_tipo_i:.6f}")

assert abs(taxa_erro_tipo_i - ALPHA) < 0.005

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(testes_h0.pvalue, bins=20, range=(0, 1), color="#4c78a8",
        edgecolor="white", density=True)
ax.axhline(1.0, color="#e45756", linestyle="--", label="uniforme ideal sob H₀")
ax.set(xlabel="p-value", ylabel="Densidade",
       title="Sob H₀, p-values contínuos são aproximadamente uniformes")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Poder: qual efeito queremos detectar?

Para um teste unilateral à esquerda com variância normal conhecida no planejamento, a distribuição da estatística sob uma média alternativa \(\mu_1\) é uma t não central. A não centralidade é

\[
\lambda=\frac{\mu_1-\mu_0}{\sigma/\sqrt n}.
\]

O poder depende da alternativa concreta: não existe “o poder do teste” sem declarar efeito, variabilidade, tamanho amostral, \(\alpha\) e direção.

In [ ]:
def poder_t_uma_amostra(mu1, mu0=100.0, sigma=5.0, n=30, alpha=0.05):
    """Poder teórico aproximado do teste t unilateral à esquerda."""
    gl = n - 1
    critico = t.ppf(alpha, df=gl)
    nao_centralidade = (mu1 - mu0) / (sigma / np.sqrt(n))
    return nct.cdf(critico, df=gl, nc=nao_centralidade)

efeitos = np.array([0.0, -1.0, -2.0, -3.0, -4.0])
tamanhos = np.array([10, 20, 30, 50, 100])
linhas = []

for n_atual in tamanhos:
    for efeito in efeitos:
        linhas.append({
            "n": n_atual,
            "diferença μ₁−μ₀ (ms)": efeito,
            "poder": poder_t_uma_amostra(100.0 + efeito, n=n_atual),
        })

tabela_poder = pd.DataFrame(linhas)
display(tabela_poder.pivot(index="diferença μ₁−μ₀ (ms)", columns="n", values="poder").round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for n_atual in tamanhos:
    parte = tabela_poder[tabela_poder["n"] == n_atual]
    ax.plot(-parte["diferença μ₁−μ₀ (ms)"], parte["poder"], marker="o", label=f"n={n_atual}")

ax.axhline(0.80, color="#e45756", linestyle="--", label="referência: 80%")
ax.set(xlabel="Redução verdadeira da latência (ms)", ylabel="Poder",
       title="Poder cresce com o efeito e com o tamanho amostral", ylim=(0, 1.02))
ax.legend(ncol=2)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### O compromisso de mudar \(\alpha\)

Mantidos \(n\), efeito e variância, aumentar \(\alpha\) facilita a rejeição: reduz \(\beta\) e aumenta o poder, mas também aumenta o erro tipo I. A decisão deve refletir o custo dos dois erros e ser feita antes da análise.

In [ ]:
alphas = np.array([0.01, 0.05, 0.10])
cenario = pd.DataFrame({
    "α = erro tipo I sob H₀": alphas,
    "poder para μ₁=98, n=30": [
        poder_t_uma_amostra(98.0, n=30, alpha=a) for a in alphas
    ],
})
display(cenario.round(4))

### Conferência de poder por Monte Carlo

A mesma definição pode ser estimada repetindo experimentos sob \(\mu_1=98\) ms. O valor simulado deve se aproximar do cálculo com a t não central.

In [ ]:
rng = np.random.default_rng(SEED + 1)
mu1 = 98.0
sigma = 5.0
n_alt = 30
repeticoes_alt = 50_000

amostras_alt = rng.normal(mu1, sigma, size=(repeticoes_alt, n_alt))
p_alt = ttest_1samp(amostras_alt, popmean=100.0, axis=1, alternative="less").pvalue
poder_simulado = np.mean(p_alt < ALPHA)
poder_teorico = poder_t_uma_amostra(mu1, sigma=sigma, n=n_alt, alpha=ALPHA)

print(f"Poder teórico:  {poder_teorico:.6f}")
print(f"Poder simulado: {poder_simulado:.6f}")
print(f"Erro absoluto:  {abs(poder_simulado - poder_teorico):.6f}")

assert abs(poder_simulado - poder_teorico) < 0.01

## 4. Comparação de modelos: preserve o pareamento

Os dois modelos são avaliados nos mesmos 120 exemplos. A dificuldade de cada exemplo afeta ambas as perdas, portanto a unidade de análise é a **diferença por exemplo**:

\[
D_i=L_{A,i}-L_{B,i}.
\]

Se valores positivos indicam que B tem menor perda, testamos \(H_1:\mu_D>0\). Tratar as duas colunas como amostras independentes descarta a correlação útil e costuma perder poder.

In [ ]:
rng = np.random.default_rng(SEED + 2)
n_exemplos = 120
dificuldade = rng.normal(0.0, 0.12, n_exemplos)
perda_a = 0.55 + dificuldade + rng.normal(0.0, 0.04, n_exemplos)
perda_b = 0.53 + dificuldade + rng.normal(0.0, 0.04, n_exemplos)
diferencas = perda_a - perda_b

pareado = ttest_rel(perda_a, perda_b, alternative="greater")
independente = ttest_ind(perda_a, perda_b, equal_var=False, alternative="greater")

comparacao = pd.DataFrame({
    "análise": ["t pareado correto", "Welch ignorando pareamento"],
    "estatística": [pareado.statistic, independente.statistic],
    "p-value": [pareado.pvalue, independente.pvalue],
})
display(comparacao.round(6))

print(f"Diferença média A−B: {diferencas.mean():.6f}")
print(f"Correlação entre perdas: {np.corrcoef(perda_a, perda_b)[0, 1]:.6f}")

assert pareado.pvalue < ALPHA
assert np.corrcoef(perda_a, perda_b)[0, 1] > 0.8

In [ ]:
ordem = np.argsort(diferencas)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axhline(0, color="black", linewidth=1)
ax.scatter(np.arange(n_exemplos), diferencas[ordem], s=22, alpha=0.75, color="#4c78a8")
ax.axhline(diferencas.mean(), color="#e45756", linestyle="--",
           label=f"diferença média = {diferencas.mean():.4f}")
ax.set(xlabel="Exemplos ordenados pela diferença", ylabel="Perda A − perda B",
       title="O teste pareado analisa uma diferença por unidade experimental")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Parada opcional: olhar repetidamente muda o teste

Agora simulamos trajetórias sob \(H_0:\mu=0\). Em um procedimento, analisamos somente em \(n=200\). No outro, começamos em \(n=20\), olhamos a cada 10 observações e paramos ao primeiro p-value bilateral menor que 0,05.

Cada olhar isolado usa 5%, mas “testar até funcionar” aumenta a probabilidade de pelo menos um falso positivo. A regra de parada faz parte do experimento.

In [ ]:
rng = np.random.default_rng(SEED + 3)
repeticoes_stop = 20_000
n_max = 200
olhares = np.arange(20, n_max + 1, 10)

trajetorias = rng.normal(0.0, 1.0, size=(repeticoes_stop, n_max))
somas = np.cumsum(trajetorias, axis=1)
somas_quadrados = np.cumsum(trajetorias**2, axis=1)

alguma_rejeicao = np.zeros(repeticoes_stop, dtype=bool)
p_final = None

for k in olhares:
    soma = somas[:, k - 1]
    soma2 = somas_quadrados[:, k - 1]
    medias = soma / k
    variancias = (soma2 - k * medias**2) / (k - 1)
    t_obs = medias / np.sqrt(variancias / k)
    p_atual = 2 * t.sf(np.abs(t_obs), df=k - 1)
    alguma_rejeicao |= p_atual < ALPHA
    if k == n_max:
        p_final = p_atual

taxa_fixa = np.mean(p_final < ALPHA)
taxa_opcional = np.mean(alguma_rejeicao)

display(pd.DataFrame({
    "regra": ["um teste em n=200", "até 19 olhares, parar se p<0,05"],
    "falso positivo": [taxa_fixa, taxa_opcional],
}).round({"falso positivo": 4}))

assert abs(taxa_fixa - ALPHA) < 0.01
assert taxa_opcional > 0.15
assert taxa_opcional > 2 * taxa_fixa

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
rotulos = ["análise fixa", "parada opcional"]
valores = [taxa_fixa, taxa_opcional]
barras = ax.bar(rotulos, valores, color=["#4c78a8", "#e45756"])
ax.axhline(ALPHA, color="black", linestyle="--", label="α nominal = 0,05")
ax.bar_label(barras, labels=[f"{v:.3f}" for v in valores], padding=3)
ax.set(ylabel="Probabilidade de falso positivo",
       title="Repetir testes sem correção infla o erro tipo I", ylim=(0, max(valores) * 1.25))
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Relatório mínimo auditável

Um resultado responsável registra:

- pergunta, população e unidade experimental;
- \(H_0\), \(H_1\), direção e \(\alpha\), todos pré-especificados;
- estatística, graus de liberdade e p-value;
- estimativa e intervalo de confiança compatível;
- regra de parada, exclusões e análises realizadas;
- limitações das hipóteses e relevância operacional.

O p-value é apenas uma peça. Na Aula 16, a estimativa será acompanhada explicitamente de tamanho de efeito.

In [ ]:
checagens = {
    "teste manual coincide com SciPy": np.isclose(t_manual, resultado.statistic, atol=1e-12),
    "erro tipo I calibrado": abs(taxa_erro_tipo_i - ALPHA) < 0.005,
    "poder teórico ≈ simulado": abs(poder_simulado - poder_teorico) < 0.01,
    "pareamento preservado": pareado.pvalue < ALPHA,
    "parada opcional infla falso positivo": taxa_opcional > 2 * taxa_fixa,
}

for nome, passou in checagens.items():
    print(f"{'✓' if passou else '✗'} {nome}")

assert all(checagens.values())
print("\nTodas as verificações passaram.")

## Desafios

1. Troque a alternativa do teste da latência para bilateral. Como mudam o p-value e a pergunta respondida?
2. Na função de poder, compare \(\sigma=5\) e \(\sigma=8\). Explique por que o poder cai.
3. Aumente a melhoria média entre os modelos, mantendo a mesma semente, e compare os testes pareado e independente.
4. Na parada opcional, reduza a frequência dos olhares. A inflação desaparece ou apenas diminui?
5. Escreva um parágrafo de resultado sem usar “aceitamos \(H_0\)” nem “há 95% de chance de \(H_0\) ser falsa”.